# Extension of continuum positions to non-epithelial cells

In [ ]:
import scanpy as sc
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import squidpy as sq
import pandas as pd
import seaborn as sns
import decoupler as dc
from scipy.spatial import cKDTree
from collections import defaultdict
import scicm
from Xenium_5K_IPF_constants import ALL_SUBTYPE_LIST, CMAP_COLORLIST, EPITHELIAL_SUBTYPE, WORK_PATH

# Option
sc.settings.verbosity = 3
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl

# Constant Declaration
dist = 30

In [ ]:
epi_cptime = pd.read_csv(f"{WORK_PATH}CSV/021_epithelial_cPosition.csv", index_col="cell_id")

In [ ]:
adata = sc.read_h5ad(f"{WORK_PATH}AnnData/spatial_merged.h5ad")
celltype = pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")
adata = adata[celltype.index,].copy()
adata.obs = pd.merge(adata.obs, celltype[["cell_type_major", "subtype"]], left_index=True, right_index=True)

In [ ]:
ne_adata = adata[~adata.obs["cell_type_major"].isin(["Epithelial"])].copy()
e_adata = adata[adata.obs["cell_type_major"].isin(["Epithelial"])].copy()

In [ ]:
ne_coords = ne_adata.obsm["spatial"]
ne_ids  = ne_adata.obs_names

In [ ]:
# Build a KD-tree on non-epithelial coordinates for fast radius queries
tree = cKDTree(ne_coords)

epi_ids = e_adata.obs_names.intersection(epi_cptime.index)

In [ ]:
# Extract spatial coordinates once and create a mapping from obs_name to row index for fast lookup
name_to_row = {name: i for i, name in enumerate(adata.obs_names)}
coords = adata.obsm["spatial"]

epi_nei_dict = {}

In [ ]:
for cid in epi_ids:
    target_coord = coords[name_to_row[cid]]

    # Indices returned here are relative to the non-epithelial subset
    idxs = tree.query_ball_point(target_coord, r=dist)

    if len(idxs) == 0:
        epi_nei_dict[cid] = {}
        continue

    # Compute distances only for neighbors within the radius (much cheaper than all-pairs distances)
    neigh_coords = ne_coords[idxs]
    dists = np.linalg.norm(neigh_coords - target_coord, axis=1)

    # Map non-epithelial indices to non-epithelial cell IDs and store distances
    epi_nei_dict[cid] = {ne_ids[i]: float(d) for i, d in zip(idxs, dists)}

In [ ]:
# Non-epithelial to epthelial continuum position and distances
ne_ids = set(ne_ids)
ne_nei_dict = defaultdict(dict)
for epi_id, neigh_dict in epi_nei_dict.items():
    for nbr_id, d in neigh_dict.items():
        if nbr_id in ne_ids:
            ne_nei_dict[nbr_id][epi_id] = float(d)

ne_nei_dict = dict(ne_nei_dict)

In [ ]:
epi_t = epi_cptime["C_position"]  # index=cell_id, column "t"

out = {}
atol = 1e-8  # tolerance for boundary equality

for ne_id, epi_di in ne_nei_dict.items():
    e_ids = np.fromiter(epi_di.keys(), dtype=object)
    dists = np.fromiter(epi_di.values(), dtype=float)
    tvals = epi_t.reindex(e_ids).to_numpy(dtype=float)

    # Priority case: if any neighbor is exactly on the boundary (d == dist),
    # use its t value and ignore all others.
    on_boundary = np.isclose(dists, dist, atol=atol)
    if np.any(on_boundary):
        boundary_idx = np.where(on_boundary)[0][0]  # assume at most one
        out[ne_id] = float(tvals[boundary_idx])
        continue

    # Otherwise, use neighbors strictly inside the radius with linear weights w = dist - d
    w = dist - dists  # >0 because d<dist here
    w_norm = w / w.sum()  # normalized so sum(w_norm)=1
    out[ne_id] = float((tvals * w_norm).sum())

ne_cptime = pd.Series(out, name="C_position")
ne_cptime.index.name = "cell_id"

ne_adata.obs["C_position"]=ne_cptime
ne_adata.obs["C_position_nan"]=np.where(ne_adata.obs["C_position"].isna(), "undefined", "defined")

In [ ]:
ne_cp_df = ne_adata[ne_adata.obs["C_position_nan"]=="defined"].obs[["C_position"]]

In [ ]:
cp_df = pd.concat([epi_cptime[["C_position"]], ne_cp_df], axis=0)

In [ ]:
cp_adata = adata[cp_df.index, ].copy()

In [ ]:
cp_adata.obs["C_position"]=cp_df[["C_position"]]

In [ ]:
cp_adata.raw = cp_adata.copy()

In [ ]:
sc.pp.normalize_total(cp_adata, target_sum=1e4)
sc.pp.log1p(cp_adata)

In [ ]:
progeny = dc.op.progeny(organism="human")

In [ ]:
dc.mt.mlm(data=cp_adata, net=progeny)


In [ ]:
score = dc.pp.get_obsm(adata=cp_adata, key="score_mlm")
pws = dc.tl.rankby_order(adata=score, order='C_position')

In [ ]:
n_bins = 20
bin_pws_dict = {
    key: pd.DataFrame(0.0, index=[n for n in range(n_bins)], columns=list(score.obs["cell_type_major"].unique())) 
    for key in list(progeny["source"].unique())
}

rej_type = []

for m_type in score.obs["cell_type_major"].unique():
    bin_temp= dc.pp.bin_order(
        adata=score[score.obs["cell_type_major"]==m_type].copy(),
        order="C_position",
        names=list(progeny["source"].unique()),
        label="C_position",
        nbins=n_bins
    )

    if n_bins != len(bin_temp["order"].unique()):
        rej_type.append(m_type)
        continue
    
    out = (
        bin_temp.groupby(["name", "order"], as_index=False)
          .agg(value_mean=("value", "mean"))
          .sort_values(["name", "order"])
    )

    for pw in list(progeny["source"].unique()):
        bin_pws_dict[pw].loc[:,m_type] = list(out[out["name"]==pw]["value_mean"])


In [ ]:
for pw in list(progeny["source"].unique()):
    for rt in rej_type:
        del bin_pws_dict[pw][rt]

In [ ]:
for pway in list(progeny["source"].unique()):
    
    fig, axes = plt.subplots(2,1,figsize=(7,4), height_ratios=(1,11))
    fig.subplots_adjust(left=0.35, right=0.83, top=0.89, bottom=0.03, hspace=0.05)
    
    v = [bin_pws_dict[pway].max().max(), bin_pws_dict[pway].min().min()]
    cl = (v[0]-v[1])*0.1
    vmax, vmin = v[0]-cl, v[1]+cl
    
    sns.heatmap(bin_pws_dict[pway].T, ax=axes[1],  cmap='scicm.PkY', cbar=None, vmax=vmax, vmin=vmin)
    ax_box = axes[1].get_position()
    
    # delete X-axis labels
    axes[1].set_xlabel("")
    axes[1].set_xticks([])
    axes[1].tick_params("y", labelsize=15)
    for n, t in enumerate(axes[1].get_yticklabels()):
        if n == 0:
            t.set(color="red", weight=500)
    
    # activity score bar
    cbar_ax = fig.add_axes([0.85, 0.2, 0.03, 0.6])
    norm_cb = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
    sm_cb = plt.cm.ScalarMappable(cmap="scicm.PkY", norm=norm_cb)
    cbar = fig.colorbar(sm_cb, cax=cbar_ax, orientation="vertical")
    cbar_ax.set_ylabel("Scaled pathway activity", fontsize=15)
    cbar_ax.tick_params("y", labelsize=14)
    
    # Continuum position bar
    norm_cp = matplotlib.colors.Normalize(vmin=0, vmax=1)
    sm_cp = plt.cm.ScalarMappable(cmap="plasma", norm=norm_cp)
    cp_bar = fig.colorbar(sm_cp, cax=axes[0], ticks=[0, 0.5, 1], orientation="horizontal")
    cp_bar.ax.set_xticklabels(['0.0', "0.5", "1.0"], fontsize=14, c="black")
    axes[0].xaxis.set_ticks_position('top')
    axes[0].set_yticks([0.5])
    axes[0].set_yticklabels(["Continuum position"], fontsize=17, weight="bold")

    label_ax = fig.add_axes([0.05, axes[1].get_position().y0, 0.1, axes[1].get_position().height], zorder=-1)
    label_ax.set_xlim(0,1)
    label_ax.set_ylim(0,1)
    label_ax.axis("off")
    label_ax.vlines(x=0.6, ymin=0.01, ymax=0.99, color="black")
    label_ax.hlines(y=0.01, xmin=0.6, xmax=1, color="black")
    label_ax.hlines(y=0.99, xmin=0.6, xmax=1, color="black")
    label_ax.text(s=f"Major cell type", x=0.5, y=0.5, fontsize=16, ha="right", va="center", rotation=90, weight="bold")

    boxdic = {
        "facecolor" : "#dddddd",
        "edgecolor" : "black",
        "boxstyle" : "Round",
        "linewidth" : 2
    }
    fig.text(x=0.2, y=0.93, s=f"{pway}", fontsize=16, weight=500, bbox=boxdic, ha="center")
    plt.savefig(f"{WORK_PATH}Figure/022_exEcNP_cPosition_projeny_mat_{pway}.png", dpi=500, format="png", bbox_inches=None)

In [ ]:
ep_score = score[score.obs["subtype"].isin(EPITHELIAL_SUBTYPE)].copy()

In [ ]:
ep_cat = []
for e in EPITHELIAL_SUBTYPE:
    for n in range(5):
        ep_cat.append(f"{e}_{n}")

In [ ]:
ep_score.obs["C_position_5"] = (ep_score.obs["C_position"]*5//1).astype(int).astype(str)
ep_score.obs.loc[ep_score.obs[ep_score.obs["C_position_5"] == "5"].index[0],"C_position_5"]="4"

In [ ]:
ep_score.obs["ep_5"]=ep_score.obs["subtype"].str.cat(ep_score.obs["C_position_5"], sep="_")
ep_score.obs["ep_5"]=pd.Categorical(ep_score.obs["ep_5"], categories=ep_cat, ordered=True)

In [ ]:
counts_5 = ep_score.obs["C_position_5"].value_counts()

In [ ]:
prop_df = pd.crosstab(
    ep_score.obs["C_position_5"],
    ep_score.obs["subtype"],
    normalize="index"
)

In [ ]:
prop_df = prop_df.reindex(index=["0", "1", "2", "3", "4"], fill_value=0)
prop_df = prop_df.reindex(columns=EPITHELIAL_SUBTYPE, fill_value=0)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(5.5, 4), height_ratios=(1,2.5))

sm_cp = plt.get_cmap("plasma", 5)

x = np.arange(len(prop_df.index))
bottom = np.ones(len(prop_df.index))
bar_w = 0.6

ax1.bar(x, counts_5, color=sm_cp.colors, width=bar_w, ec="black", linewidth=0.5)

for subtype in prop_df.columns:
    y = prop_df[subtype].values
    ax2.bar(x, y, bottom=bottom-y, width=bar_w, label=subtype, ec="black", linewidth=0.5)
    bottom -= y

cum_df = prop_df.cumsum(axis=1)
line_x = []
for c in x:
    line_x.append(c-bar_w/2)
    line_x.append(c+bar_w/2)

for i, subtype in enumerate(cum_df.columns[:-1]): 
    y = cum_df[subtype].values
    line_y = [1 - c for c in y for _ in range(2)]
    ax2.plot(
        line_x, line_y,
        linestyle="--",
        linewidth=0.5,
        color="black",
        zorder=-1
    )

ax1.set_xticks(x, labels=[])
ax1.set_ylabel("Cell count", fontsize=12)
ax1.set_yticks(ticks=[0, 5e3, 10e3, 15e3], labels=["0", "5K", "10K", "15K"])
ax2.set_xticks(x, labels=["[{:.1f}-{:.1f}]".format(t/5,t/5+0.2) for t in range(5)])
ax2.set_xlabel("Continuum position interval", fontsize=12)
ax2.set_yticks([0, 0.25, 0.5, 0.75, 1], labels=["0", "25", "50", "75", "100"])
ax2.set_ylabel("Proportion (%)", fontsize=12)
ax2.set_ylim(0, 1)

# 凡例
ax2.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)

for ax in [ax1, ax2]:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()

plt.savefig(f"{WORK_PATH}Figure/022_EcNP_cPosition_interval_count_proporition.png", dpi=500, format="png", bbox_inches=None)
plt.show()

In [ ]:
ep_df = ep_score.to_df()

In [ ]:
ep_df = pd.merge(ep_df.copy(), ep_score.obs[["subtype","C_position_5"]], left_index=True, right_index=True)

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(4,7))
axes = axes.flatten()
fig.subplots_adjust(hspace=0.9, wspace=0.3, top=0.95, bottom=0.2, left=0.3, right=0.85)
pa=["Androgen", "Hypoxia"]
mask = (prop_df<0.03).T

# NA array
nan_x = []
nan_y = []
for n, (_, v) in enumerate(mask.iterrows()):
    for m in range(5):
        if v.iloc[m]:
            nan_x.append(m)
            nan_y.append(n)

# heatmap
for ax,p in zip(axes,pa):
    temp_df = (ep_df[[p,"subtype","C_position_5"]]
               .groupby(["subtype","C_position_5"], observed=False, as_index=False)
               .mean()
               .pivot_table(index="C_position_5", columns="subtype", values=p, observed=False)
              )
    temp_df = temp_df.reindex(index=["0", "1", "2", "3", "4"], fill_value=0)
    temp_df = temp_df.reindex(columns=EPITHELIAL_SUBTYPE, fill_value=0).T
    sns.heatmap(
        data=temp_df,
        cmap="coolwarm",
        mask=mask,
        cbar_kws={"label":"Pathway-wise scaled\nmean activity score"},
        linewidths=0.5,
        linecolor="black",
        ax=ax
    )
    ax.set_facecolor("#444444")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(f"{p}", fontsize=13, weight=500)
    ax.set_xticks([])
    ax.spines.bottom.set_visible(True)
    ax.spines.right.set_visible(True)
    
    cpi_ax = fig.add_axes([ax.get_position().x0, ax.get_position().y0-0.025, ax.get_position().width, 0.02])
    cpi_ax.set_ylim(0,1)
    cpi_ax.set_xlim(0,1)
    cpi_ax.set_yticks([])
    cpi_ax.set_xlabel("Continuum position interval", fontsize=12)
    cpi_ax.set_xticks(ticks=[0.1+0.2*n for n in range(5)], 
                      labels=["[{:.1f}-{:.1f}]".format(0.2*n, 0.2*(n+1)) for n in range(5)], 
                      rotation=55, rotation_mode="anchor", va="top",ha="right")

    for n, c in enumerate(sm_cp.colors):
        r = matplotlib.patches.Rectangle(xy=(n*0.2, 0), width=0.2, height=1, fc=c, fill=True, ec="black", linewidth=0.5)
        cpi_ax.add_patch(r)

    for x,y in zip(nan_x,nan_y):
        ax.text(s="NA", x=x+0.5,y=y+0.5, ha="center", va="center", color="white")

plt.savefig(f"{WORK_PATH}Figure/022_exEcNP_cPosition_projeny_epithelial_5bin.png", dpi=500, format="png", bbox_inches=None)